# TMJDLC — Javtiful Telegram Bot

Google Colab runner for the Javtiful downloader. Uses Neon PostgreSQL, a main bot, a WZGram video upload session, optional WZML-X-style helper upload bots, and a separate screenshot bot.

Batch pipeline: **1 downloader + 1 uploader**, with **1 ready video** buffered by default. Run cells in order. Restart the runtime and rerun the notebook after code or secret changes.


## 1. Colab Secrets

Add these secrets and enable notebook access:

- `TG_BOT_TOKEN` — main bot token
- `TG_SCREENSHOT_BOT_TOKEN` — screenshot bot token
- `TG_API_ID` and `TG_API_HASH` — Telegram API credentials
- `TG_ADMIN_IDS` — admin Telegram user ID(s), comma-separated
- `DATABASE_URL` — Neon PostgreSQL connection string
- `TG_HELPER_BOT_TOKENS` — optional comma/space-separated helper bot tokens for video-channel uploads

Each helper bot must be an administrator in the video destination channel with permission to post. Helpers are optional; the main uploader works without them. They are rotated across eligible uploads, not used to split one video into parallel chunks.

Do not put credentials in code cells.

Each helper bot must be an administrator in the video destination channel with permission to post. Helpers are optional; the main uploader works without them. They are rotated across eligible uploads, not used to split one video into parallel chunks.\n\nAfter startup, send `/start` to **both bots**:
- Main bot: configure the video destination.
- Screenshot bot: configure the screenshot destination.

For public channels, send `@channelusername`. For private channels, add the relevant bot as an administrator with posting permission, then forward a recent channel post directly to that bot.

Change destinations later with `/setupload` in the main bot and `/setchannel` in the screenshot bot.


In [ ]:
#@title 2. Sync latest TMJDLC source
%cd /content
import os, subprocess
repo = '/content/TMJDLC'
if os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git', '-C', repo, 'fetch', '--prune', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', repo, 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', repo, 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', 'https://github.com/Telebotfaroff/TMJDLC.git', repo], check=True)
%cd /content/TMJDLC
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'✅ TMJDLC source synced to main: {commit}')
subprocess.run(['git', 'status', '--short'], check=True)


In [ ]:
#@title 3. Install dependencies (run once per fresh Colab runtime)
%cd /content/TMJDLC
import subprocess, sys
print('Installing TMJDLC and Python dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
print('Installing FFmpeg and aria2...')
subprocess.run(['bash', '-lc', 'apt-get update -qq && apt-get install -y ffmpeg aria2'], check=True)
print('Installing Playwright Chromium and system dependencies...')
subprocess.run([sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'], check=True)
subprocess.run([sys.executable, '-m', 'compileall', '-q', 'tmjdlc'], check=True)
print('✅ Dependencies installed and Python modules compiled.')


In [ ]:
#@title 4. Load credentials and configure worker performance
import os
from google.colab import userdata

SECRET_NAMES = [
    'TG_BOT_TOKEN',
    'TG_SCREENSHOT_BOT_TOKEN',
    'TG_API_ID',
    'TG_API_HASH',
    'TG_ADMIN_IDS',
    'DATABASE_URL',
]
missing = []
for key in SECRET_NAMES:
    try:
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        missing.append(key)
    else:
        os.environ[key] = str(value).strip()
if missing:
    raise RuntimeError('Missing Colab Secrets: ' + ', '.join(missing))

# Optional WZML-X-style helper uploader bot tokens.
try:
    helper_tokens = userdata.get('TG_HELPER_BOT_TOKENS')
except Exception:
    helper_tokens = None
if helper_tokens:
    os.environ['TG_HELPER_BOT_TOKENS'] = str(helper_tokens).strip()
else:
    os.environ.pop('TG_HELPER_BOT_TOKENS', None)

MAX_PARALLEL = 1 #@param {type:'integer'}
UPLOAD_WORKERS = 100 #@param {type:'slider', min:1, max:100, step:1}
SCREENSHOT_UPLOAD_WORKERS = 2 #@param {type:'slider', min:1, max:8, step:1}
DOWNLOAD_FRAGMENTS = 8 #@param {type:'slider', min:1, max:16, step:1}
MIN_FREE_DISK_GB = 3.0 #@param {type:'number'}
AUTO_CLEAN = True #@param {type:'boolean'}
NOTIFY_PROGRESS_STEP = 10 #@param {type:'integer'}
QUEUE_POLL_SECONDS = 4 #@param {type:'integer'}
MAX_PENDING_UPLOADS = 1 #@param {type:'slider', min:0, max:4, step:1}
UPLOAD_QUEUE_POLL_SECONDS = 2 #@param {type:'integer'}
JOB_MAX_ATTEMPTS = 3 #@param {type:'integer'}
JOB_LEASE_SECONDS = 3600 #@param {type:'integer'}
TMJDLC_UPLOAD_FLOOD_RETRIES = 5 #@param {type:'slider', min:1, max:10, step:1}

if MIN_FREE_DISK_GB < 0 or NOTIFY_PROGRESS_STEP < 1 or MAX_PENDING_UPLOADS < 0 or UPLOAD_QUEUE_POLL_SECONDS < 1:
    raise ValueError('Disk threshold must be non-negative and progress step must be at least 1.')

os.environ.update({
    'TMJDLC_MAX_PARALLEL': str(MAX_PARALLEL),
    'TMJDLC_UPLOAD_WORKERS': str(UPLOAD_WORKERS),
    'TMJDLC_UPLOAD_FLOOD_RETRIES': str(TMJDLC_UPLOAD_FLOOD_RETRIES),
    'TMJDLC_SCREENSHOT_UPLOAD_WORKERS': str(SCREENSHOT_UPLOAD_WORKERS),
    'TMJDLC_DOWNLOAD_FRAGMENTS': str(DOWNLOAD_FRAGMENTS),
    'TMJDLC_USE_ARIA2': 'true',
    'TMJDLC_MIN_FREE_DISK_GB': str(MIN_FREE_DISK_GB),
    'TMJDLC_AUTO_CLEAN': str(AUTO_CLEAN).lower(),
    'TMJDLC_NOTIFY_PROGRESS_STEP': str(NOTIFY_PROGRESS_STEP),
    'TMJDLC_QUEUE_POLL_SECONDS': str(QUEUE_POLL_SECONDS),
    'TMJDLC_MAX_PENDING_UPLOADS': str(MAX_PENDING_UPLOADS),
    'TMJDLC_UPLOAD_QUEUE_POLL_SECONDS': str(UPLOAD_QUEUE_POLL_SECONDS),
    'TMJDLC_JOB_MAX_ATTEMPTS': str(JOB_MAX_ATTEMPTS),
    'TMJDLC_JOB_LEASE_SECONDS': str(JOB_LEASE_SECONDS),
    'TMJDLC_WORK_DIR': '/content/tmjdlc-bot',
})
os.makedirs(os.environ['TMJDLC_WORK_DIR'], exist_ok=True)

print('✅ Credentials loaded (values hidden).')
print(
    f'Pipeline workers: 1 downloader + 1 video uploader | WZGram transmission slots/client: {UPLOAD_WORKERS} '
    f'| Screenshot upload transmissions: {SCREENSHOT_UPLOAD_WORKERS} '
    f'| Download fragments: {DOWNLOAD_FRAGMENTS}'
)
print(f'Pause below: {MIN_FREE_DISK_GB} GiB free | Retry limit: {JOB_MAX_ATTEMPTS} | Auto-clean: {AUTO_CLEAN}')
print(f'Progress updates: every {NOTIFY_PROGRESS_STEP}% | Queue poll: {QUEUE_POLL_SECONDS}s')
print(f'Upload buffer: {MAX_PENDING_UPLOADS} ready video(s) in addition to the active upload | Pause/resume poll: {UPLOAD_QUEUE_POLL_SECONDS}s')
helper_count = len([x for x in os.getenv('TG_HELPER_BOT_TOKENS', '').replace(',', ' ').split() if x])
print(f'FloodWait retry limit: {TMJDLC_UPLOAD_FLOOD_RETRIES} | Helper uploader bots configured: {helper_count}')


In [ ]:
#@title 5. Preflight checks
import shutil, importlib.util, sys
if sys.version_info < (3, 12):
    raise RuntimeError(f'TMJDLC requires Python 3.12+. Current runtime: {sys.version.split()[0]}')
for executable in ('ffmpeg', 'ffprobe', 'aria2c'):
    if shutil.which(executable) is None:
        raise RuntimeError(f'{executable} is not installed or not on PATH.')
for module in ('pyrogram', 'psycopg', 'playwright', 'tmjdlc'):
    if importlib.util.find_spec(module) is None:
        raise RuntimeError(f'Required Python module is missing: {module}')
disk = shutil.disk_usage(os.environ['TMJDLC_WORK_DIR'])
print(f'Python: {sys.version.split()[0]} | FFmpeg/FFprobe/aria2: ready')
print(f'Free Colab disk: {disk.free / 1024**3:.1f} GiB')
print('✅ Preflight passed. Neon schema initialization and expired-job recovery run at bot startup.')


## 6. Start the bot

Run the next cell and keep it running. Send `/start` to both bots and configure their destinations when prompted.

The pipeline uses one downloader and one video uploader. It keeps one additional completed video ready for upload, then pauses downloading until capacity is available. Queue state is stored in Neon; Colab media files are temporary.


In [ ]:
#@title 7. Start TMJDLC — WZGram MTProto
%cd /content/TMJDLC
print('Starting TMJDLC. Keep this cell running while the bot is in use.')
!bash -o pipefail -c 'PYTHONUNBUFFERED=1 python -u -m tmjdlc.bot 2>&1 | tee -a /content/tmjdlc-bot/bot.log'


## Telegram controls

- `/start`, `/panel` — dashboard
- **Download Video** — download one Javtiful URL (720p)
- **Batch Download** — process channel pages
- **Queue Manager** — pause, resume, cancel, retry, or clear jobs
- **Settings** — screenshots, destinations, worker settings
- **Storage / Worker Status** — disk and pipeline status

Commands: `/dl URL`, `/batch CHANNEL_URL all`, `/batch CHANNEL_URL 1-5,8`, `/queue`, `/settings`, `/storage`, `/cancel JOB_ID`, `/retry JOB_ID`.

### Pipeline and storage

One downloader prepares the next video while one video uploads. With `TMJDLC_MAX_PENDING_UPLOADS=1`, one extra video can wait in `ready` state. Set it to `0` to disable that buffer. The downloader resumes automatically when upload capacity opens.

### Screenshot bot

Set `TG_SCREENSHOT_BOT_TOKEN` in Colab Secrets. Add the screenshot bot as an administrator with photo-posting permission. Configure its destination through `/start` or `/setchannel`. Use a different token from the main bot.
